In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Set the source path of the zip file in Google Drive
drive_zip_path = '/content/drive/MyDrive/archive.zip'

In [6]:
import shutil

# Set the destination path to extract the contents
colab_extract_path = '/content/data'

# Extract the zip file to the destination path
shutil.unpack_archive(drive_zip_path, colab_extract_path)

In [14]:
import sys
import os
import pandas as pd
import numpy as np
from keras.models import Sequential
from keras.layers import (
    Dense,
    Dropout,
    Activation,
    Flatten,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
)
from keras.losses import categorical_crossentropy
from keras.optimizers import Adam
from keras.regularizers import l2
from keras.utils import np_utils
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

df = pd.read_csv('/content/data/fer2013.csv')

X_train, train_y, X_test, test_y = [], [], [], []

for index, row in df.iterrows():
    val = row['pixels'].split(" ")
    try:
        if 'Training' in row['Usage']:
            X_train.append(np.array(val, 'float32'))
            train_y.append(row['emotion'])
        elif 'PublicTest' in row['Usage']:
            X_test.append(np.array(val, 'float32'))
            test_y.append(row['emotion'])
    except:
        print(f"error occurred at index: {index} and row: {row}")

num_labels = 7
batch_size = 64
epochs = 150
width, height = 48, 48

X_train = np.array(X_train, 'float32')
train_y = np.array(train_y, 'float32')
X_test      = np.array(X_test, 'float32')
test_y = np.array(test_y, 'float32')

train_y = np_utils.to_categorical(train_y, num_classes=num_labels)
test_y = np_utils.to_categorical(test_y, num_classes=num_labels)

# Normalizing data between 0 and 1
X_train /= 255.0
X_test /= 255.0

X_train = X_train.reshape(X_train.shape[0], width, height, 1)
X_test = X_test.reshape(X_test.shape[0], width, height, 1)

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)

# Model architecture
model = Sequential()

model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(width, height, 1)))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
model.add(Conv2D(256, kernel_size=(3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(512, kernel_size=(3, 3), activation='relu'))
model.add(Conv2D(512, kernel_size=(3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(GlobalAveragePooling2D())

model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(num_labels, activation='softmax'))

# Compile the model
model.compile(loss=categorical_crossentropy,
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])

# Set up callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, verbose=1)

# Data augmentation and training
model.fit(datagen.flow(X_train, train_y, batch_size=batch_size),
                    steps_per_epoch=len(X_train) / batch_size,
                    epochs=epochs,
                    validation_data=(X_test, test_y),
                    callbacks=[early_stopping, reduce_lr])


# Saving the model to use it later on
fer_json = model.to_json()
with open("fer.json", "w") as json_file:
    json_file.write(fer_json)
model.save_weights("fer.h5")


Epoch 1/150
448/448 [==============================] - 25s 50ms/step - loss: 2.2426 - accuracy: 0.1976 - val_loss: 2.0503 - val_accuracy: 0.2533 - lr: 0.0010
Epoch 2/150
448/448 [==============================] - 22s 48ms/step - loss: 1.8776 - accuracy: 0.2450 - val_loss: 1.7705 - val_accuracy: 0.2781 - lr: 0.0010
Epoch 3/150
448/448 [==============================] - 20s 45ms/step - loss: 1.7385 - accuracy: 0.3038 - val_loss: 2.1190 - val_accuracy: 0.2123 - lr: 0.0010
Epoch 4/150
448/448 [==============================] - 22s 50ms/step - loss: 1.6151 - accuracy: 0.3673 - val_loss: 1.5051 - val_accuracy: 0.4419 - lr: 0.0010
Epoch 5/150
448/448 [==============================] - 21s 46ms/step - loss: 1.5191 - accuracy: 0.4080 - val_loss: 1.4915 - val_accuracy: 0.3968 - lr: 0.0010
Epoch 6/150
448/448 [==============================] - 21s 47ms/step - loss: 1.4628 - accuracy: 0.4343 - val_loss: 1.3536 - val_accuracy: 0.5001 - lr: 0.0010
Epoch 7/150
448/448 [==============================]